In [1]:
pip install torch torchvision


Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import os


In [23]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

# ======================================
# 1️⃣ Data Processing - Custom Dataset
# ======================================
class WasteDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.image_files = [f for f in os.listdir(img_dir) if f.endswith('.jpg')]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        label_path = os.path.join(self.label_dir, img_name.replace('.jpg', '.txt'))

        image = Image.open(img_path).convert("RGB")
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                class_idx = int(f.readline().strip().split()[0])
        else:
            raise ValueError(f"Label file not found for image: {img_path}")
        
        if self.transform:
            image = self.transform(image)

        return image, class_idx

# Define transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Paths (CHANGE THESE)
train_images = r"C:\Users\Alfina\Downloads\archive\Warp-D\train\images"
train_labels = r"C:\Users\Alfina\Downloads\archive\Warp-D\train\labels"
test_images = r"C:\Users\Alfina\Downloads\archive\Warp-D\test\images"
test_labels = r"C:\Users\Alfina\Downloads\archive\Warp-D\test\labels"

# Load datasets
train_dataset = WasteDataset(train_images, train_labels, transform)
test_dataset = WasteDataset(test_images, test_labels, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ======================================
# 2️⃣ Load Pretrained VGG16 & Fine-tune
# ======================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.vgg16(pretrained=True)

# Freeze feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Modify classifier for 28 classes
num_features = model.classifier[6].in_features
model.classifier[6] = nn.Linear(num_features, 28)  # 28 classes
model = model.to(device)

# ======================================
# 3️⃣ Hyperparameter Tuning
# ======================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.0005)
early_stopping_patience = 5
num_epochs = 10

# ======================================
# 4️⃣ Training & Evaluation
# ======================================
best_accuracy = 0.0
early_stop_count = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_accuracy = 100 * correct / total
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(train_loader):.4f} | Accuracy: {train_accuracy:.2f}%")

    # Validation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    test_accuracy = 100 * correct / total
    print(f"Validation Accuracy: {test_accuracy:.2f}%")

    # Early stopping
    if test_accuracy > best_accuracy:
        best_accuracy = test_accuracy
        early_stop_count = 0
        torch.save(model.state_dict(), r"C:\wasteproject\vgg16_waste_best.pth")
        print("✅ Model Saved!")
    else:
        early_stop_count += 1

    if early_stop_count >= early_stopping_patience:
        print("🛑 Early stopping triggered!")
        break



Epoch 1/10 | Loss: 3.3405 | Accuracy: 6.81%
Validation Accuracy: 8.43%
✅ Model Saved!
Epoch 2/10 | Loss: 3.1960 | Accuracy: 8.77%
Validation Accuracy: 5.75%
Epoch 3/10 | Loss: 3.1441 | Accuracy: 11.17%
Validation Accuracy: 6.13%
Epoch 4/10 | Loss: 3.0050 | Accuracy: 14.93%
Validation Accuracy: 9.58%
✅ Model Saved!
Epoch 5/10 | Loss: 2.6922 | Accuracy: 23.37%
Validation Accuracy: 8.43%
Epoch 6/10 | Loss: 2.1514 | Accuracy: 36.95%
Validation Accuracy: 9.39%
Epoch 7/10 | Loss: 1.5236 | Accuracy: 56.08%
Validation Accuracy: 8.62%
Epoch 8/10 | Loss: 1.0857 | Accuracy: 68.39%
Validation Accuracy: 10.34%
✅ Model Saved!
Epoch 9/10 | Loss: 0.8074 | Accuracy: 76.35%
Validation Accuracy: 7.47%
Epoch 10/10 | Loss: 0.5841 | Accuracy: 82.71%
Validation Accuracy: 6.51%


In [38]:
print("Loaded Class Names:", class_names)


Loaded Class Names: ['bottle-blue', 'bottle-green', 'bottle-dark', 'bottle-milk', 'bottle-transp', 'bottle-multicolor', 'bottle-yogurt', 'bottle-oil', 'cans', 'juice-cardboard', 'milk-cardboard', 'detergent-color', 'detergent-transparent', 'detergent-box', 'canister', 'bottle-blue-full', 'bottle-transp-full', 'bottle-dark-full', 'bottle-green-full', 'bottle-multicolorv-full', 'bottle-milk-full', 'bottle-oil-full', 'detergent-white', 'bottle-blue5l', 'bottle-blue5l-full', 'glass-transp', 'glass-dark', 'glass-green']


In [46]:
import torch
import torch.nn as nn
import yaml  # Import PyYAML to read dataset.yaml
from torchvision import models, transforms
from PIL import Image

# 🔹 Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 🔹 Load Class Names from dataset.yaml
dataset_yaml_path = r"C:\Users\Alfina\Downloads\archive\Warp-D\dataset.yaml"
with open(dataset_yaml_path, "r") as file:
    dataset_config = yaml.safe_load(file)

class_names = dataset_config["names"]  # Get class names from 'names' key
num_classes = len(class_names)  # Get total number of classes

# 🔹 Define Model Architecture (Must Match Training)
model = models.vgg16(pretrained=False)  # Load base VGG16 model
model.classifier[6] = nn.Linear(4096, num_classes)  # Modify last layer for classification

# 🔹 Load Model Weights
model.load_state_dict(torch.load(r"C:\wasteproject\vgg16_waste_best.pth", map_location=device))
model.to(device)
model.eval()

# 🔹 Define Transform (Must Match Training)
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Adjusted to 256x256
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 🔹 Multi-Label Prediction Function
def predict_multi_label(image_path, model, transform, threshold=0.1):
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        probabilities = torch.sigmoid(output).cpu().numpy()[0]  # Multi-label: use sigmoid

    predictions = [(class_names[i], prob) for i, prob in enumerate(probabilities) if prob > threshold]
    return predictions

# 🔹 Predict on a Test Image
image_path = r"C:\Users\Alfina\Downloads\archive\Warp-D\test\images\prepared_data_all_MGS_19-Oct_19-01-15.jpg"
predictions = predict_multi_label(image_path, model, transform, threshold=0.1)

# 🔹 Print Results
if predictions:
    print("🔹 Detected Classes:")
    for class_name, prob in predictions:
        print(f"➡ {class_name}: {prob:.2f}")
else:
    print("⚠ No class detected (Try lowering the threshold)")


⚠ No class detected (Try lowering the threshold)
